<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.1-B — AUTONOMOUS, INTERPOLATION ONLY, SPARSE-LU ONLY
#
# Training N : 20, 50, 100, 200, 500
# Test N     : 35, 75, 150, 350  (INTERPOLATION ONLY)
# R          : 10, 20, 50, 100, 200, 400, 800
# i0         : explicit i0=1 + varying i0>1
#
# NO inverse of T or D0 is ever formed.
#
# Extinction moments:
#       (-T)m1 = 1
#       (-T)m2 = 2m1
#
# solved by sparse LU. If double precision cannot resolve an admissible extinction
# target, that target is marked unavailable; the configuration is STILL retained for
# the exact infection-count distribution and distributional training.
# =====================================================================================

from __future__ import annotations
import copy, hashlib, json, math, pickle, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Tuple

import numpy as np
from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

import torch
import torch.nn as nn
import matplotlib.pyplot as plt


# =====================================================================================
# 1. CONFIG
# =====================================================================================

@dataclass
class Config:
    seed:int=20260819
    beta_range:Tuple[float,float]=(.30,1.50)
    gamma_range:Tuple[float,float]=(.20,1.00)
    omega_range:Tuple[float,float]=(.02,.50)
    initial_fraction_range:Tuple[float,float]=(.02,.20)
    i0_one_fraction:float=.25

    train_N:Tuple[int,...]=(20,50,100,200,500)
    interp_N:Tuple[int,...]=(35,75,150,350)

    n_train:int=800
    n_val:int=100
    n_test_seen:int=100
    n_test_interp:int=80

    width:int=128
    depth:int=3
    batch_size:int=32
    epochs:int=500
    lr:float=1e-3
    weight_decay:float=1e-6
    lambda_tau:float=.02
    patience:int=50
    min_delta:float=1e-6
    grad_clip:float=5.

    prob_tol:float=1e-10
    var_rel_tol:float=1e-8
    kl_eps:float=1e-12
    refine_steps:int=3

    teacher_version:str="sparseLU_masked_tau_B_v8"
    output_dir:str="results_section_5_1B_sparseLU"
    dpi:int=300

cfg=Config()

R_VALUES=(10,20,50,100,200,400,800)
REPEATS=3
GALLERY_R=(10,50,200,800)


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

seed_all(cfg.seed)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
out=Path(cfg.output_dir); out.mkdir(parents=True,exist_ok=True)

tag=hashlib.sha1(
    json.dumps(asdict(cfg),sort_keys=True).encode()
).hexdigest()[:12]

N_scale=max(cfg.train_N)

print("="*108)
print("EXPERIMENT 5.1-B — SPARSE-LU INTERPOLATION STUDY")
print("="*108)
print("Device:",device)
print("Training N:",cfg.train_N)
print("Held-out interpolation N:",cfg.interp_N)
print("R:",R_VALUES," | repetitions:",REPEATS)
print(f"N=500 transient states: {500*501//2:,}")
print("NO MATRIX INVERSE IS COMPUTED.")


# =====================================================================================
# 2. EXACT SIRS CTMC
# =====================================================================================

def state_space(N):
    st=[(s,i) for i in range(1,N+1) for s in range(N-i+1)]
    return st,{x:j for j,x in enumerate(st)}


def matrices(beta,gamma,omega,N):
    st,idx=state_space(N); M=len(st)
    tr=[];tc=[];tv=[]; ir=[];ic=[];iv=[]
    q=np.zeros(M,dtype=np.float64)

    for row,(s,i) in enumerate(st):
        r=N-s-i; total=0.

        if s:
            rate=beta*s*i/N; col=idx[(s-1,i+1)]
            tr.append(row);tc.append(col);tv.append(rate)
            ir.append(row);ic.append(col);iv.append(rate)
            total+=rate

        rate=gamma*i
        if i==1: q[row]+=rate
        else:
            col=idx[(s,i-1)]
            tr.append(row);tc.append(col);tv.append(rate)
        total+=rate

        if r:
            rate=omega*r; col=idx[(s+1,i)]
            tr.append(row);tc.append(col);tv.append(rate)
            total+=rate

        tr.append(row);tc.append(row);tv.append(-total)

    T=sparse.coo_matrix((tv,(tr,tc)),shape=(M,M),dtype=np.float64).tocsc()
    D1=sparse.coo_matrix((iv,(ir,ic)),shape=(M,M),dtype=np.float64).tocsc()

    return T,(T-D1).tocsc(),D1,q,idx


# =====================================================================================
# 3. SPARSE LU SOLVES ONLY
# =====================================================================================

def factor(A,ordering="COLAMD"):
    return splu(A.tocsc(),permc_spec=ordering)


def solve_lu(A,lu,b,transpose=False):
    """
    Solve Ax=b (or A^T x=b) from a sparse LU factorization,
    followed by a few iterative-refinement steps.
    """
    b=np.asarray(b,dtype=np.float64)
    mode="T" if transpose else "N"
    x=lu.solve(b,trans=mode)

    for _ in range(cfg.refine_steps):
        residual=b-(A.T@x if transpose else A@x)

        if not np.all(np.isfinite(residual)):
            break

        rel=np.linalg.norm(residual,np.inf)/max(np.linalg.norm(b,np.inf),1.)

        if rel<1e-11:
            break

        x+=lu.solve(residual,trans=mode)

    return np.asarray(x,dtype=np.float64)


# =====================================================================================
# 4. EXTINCTION MOMENTS
# =====================================================================================

def variance_linear_system(T,q,A,lu,m1):
    """
    Numerically stable identity for exactly the same Var(tau).
    It uses another sparse linear solve; no inverse is formed.
    """
    C=T.tocoo()
    keep=C.row!=C.col
    rr,cc,a=C.row[keep],C.col[keep],C.data[keep]

    source=np.bincount(
        rr,
        weights=a*(m1[cc]-m1[rr])**2,
        minlength=T.shape[0]
    ).astype(np.float64)

    source+=q*m1*m1

    if not np.all(np.isfinite(source)) or source.min()<-1e-8:
        raise ArithmeticError("Invalid variance-system RHS.")

    return solve_lu(A,lu,np.maximum(source,0.))


def extinction_moments(T,q,initial):
    """
    Original moment equations:
        (-T)m1 = 1
        (-T)m2 = 2m1

    Alternative sparse orderings are attempted when the first LU solve is
    numerically inadmissible.
    """
    A=(-T).tocsc()
    one=np.ones(A.shape[0],dtype=np.float64)
    errors=[]

    for ordering in ("COLAMD","MMD_AT_PLUS_A","NATURAL"):

        try:
            lu=factor(A,ordering)

            # E(tau)
            m1=solve_lu(A,lu,one)

            if not np.all(np.isfinite(m1)):
                raise ArithmeticError("non-finite m1 vector")

            mean=float(m1[initial])

            if mean<=0:
                raise ArithmeticError(f"E(tau)={mean}")

            # E(tau^2)
            m2=2.*solve_lu(A,lu,m1)

            if not np.all(np.isfinite(m2)):
                raise ArithmeticError("non-finite m2 vector")

            second=float(m2[initial])

            if second<=0:
                raise ArithmeticError(f"E(tau^2)={second}")

            # Original variance formula first.
            raw=(
                np.longdouble(second)
                -
                np.longdouble(mean)**2
            )

            tol=cfg.var_rel_tol*max(abs(second),mean*mean,1.)

            if np.isfinite(raw) and raw>=-tol:
                var=max(float(raw),0.)
                variance_method="moment subtraction"

            else:
                vv=variance_linear_system(T,q,A,lu,m1)
                var=float(vv[initial])
                variance_method="variance linear system"

            if not np.isfinite(var) or var<0:
                raise ArithmeticError(f"Var(tau)={var}")

            return (
                mean,second,var,True,
                f"{ordering}; {variance_method}"
            )

        except Exception as e:
            errors.append(f"{ordering}: {e}")

    # Do NOT fabricate, clip, or replace an unresolved target.
    return (
        np.nan,np.nan,np.nan,False,
        " | ".join(errors)
    )


# =====================================================================================
# 5. EXACT TEACHER
# =====================================================================================

def exact_targets(beta,gamma,omega,N,i0):

    T,D0,D1,q,idx=matrices(beta,gamma,omega,N)
    M=T.shape[0]; initial=idx[(N-i0,i0)]

    # -------------------------------------------------------------------------
    # Exact infection-count distribution.
    # Solve systems with -D0 using sparse LU.
    # -------------------------------------------------------------------------

    alpha=np.zeros(M); alpha[initial]=1.
    A0=(-D0).tocsc()

    lu0=factor(A0,"COLAMD")
    b=solve_lu(A0,lu0,q)

    v=alpha.copy()
    p=np.zeros(N+2)

    for k in range(N+1):
        p[k]=v@b

        y=solve_lu(
            A0,lu0,v,
            transpose=True
        )

        v=np.asarray(D1.T@y).ravel()

    p[-1]=v.sum()
    p[np.abs(p)<cfg.prob_tol]=0.

    if not np.all(np.isfinite(p)) or p.min()<-cfg.prob_tol:
        raise RuntimeError(f"Invalid exact PMF: N={N}, i0={i0}")

    p=np.maximum(p,0.)
    mass=p.sum()

    if not np.isfinite(mass) or abs(mass-1.)>1e-5:
        raise RuntimeError(f"Invalid PMF mass={mass}: N={N}, i0={i0}")

    p/=mass

    # -------------------------------------------------------------------------
    # Extinction-time targets.
    # -------------------------------------------------------------------------

    mean,second,var,tau_valid,tau_info=extinction_moments(
        T,q,initial
    )

    return p,mean,second,var,tau_valid,tau_info


# =====================================================================================
# 6. EXACT DATA
# =====================================================================================

@dataclass
class Record:
    beta:float; gamma:float; omega:float
    N:int; i0:int
    p:np.ndarray

    mean_tau:float
    second_tau:float
    var_tau:float

    tau_valid:bool
    tau_info:str


def scale(u,ab):
    a,b=ab
    return a+(b-a)*u


def design(n,Ns,seed):
    U=qmc.LatinHypercube(d=4,seed=seed).random(n)

    b=scale(U[:,0],cfg.beta_range)
    g=scale(U[:,1],cfg.gamma_range)
    w=scale(U[:,2],cfg.omega_range)
    f=scale(U[:,3],cfg.initial_fraction_range)

    Nv=np.tile(np.asarray(Ns),math.ceil(n/len(Ns)))[:n]
    rng=np.random.default_rng(seed+991)
    rng.shuffle(Nv)

    # Non-i0=1 stratum
    i0=np.array([
        int(np.clip(round(f[j]*Nv[j]),2,Nv[j]))
        for j in range(n)
    ])

    # Explicit i0=1 stratum
    for N in Ns:
        ix=np.where(Nv==N)[0]
        k=max(1,int(round(cfg.i0_one_fraction*len(ix))))
        i0[rng.choice(ix,k,replace=False)]=1

    return [
        (
            float(b[j]),float(g[j]),float(w[j]),
            int(Nv[j]),int(i0[j])
        )
        for j in range(n)
    ]


def make_dataset(configs,name):
    ans=[]
    unresolved=0
    t0=time.perf_counter()

    for j,(b,g,w,N,i0) in enumerate(configs,1):

        try:
            p,m,m2,v,ok,info=exact_targets(b,g,w,N,i0)

        except Exception as e:
            # Distributional exact teacher itself must remain valid.
            raise RuntimeError(
                f"\nEXACT DISTRIBUTIONAL TEACHER FAILED — {name}, record {j}\n"
                f"N={N}, i0={i0}, beta={b:.8g}, "
                f"gamma={g:.8g}, omega={w:.8g}\n{e}"
            ) from e

        unresolved+=int(not ok)

        ans.append(
            Record(
                b,g,w,N,i0,p,
                m,m2,v,
                ok,info
            )
        )

        if j%5==0 or j==len(configs):
            print(
                f"[{name:12s}] {j:4d}/{len(configs):4d} | "
                f"N={N:3d}, i0={i0:3d} | "
                f"tau unresolved={unresolved:3d} | "
                f"{time.perf_counter()-t0:.1f}s"
            )

    return ans


def load_or_make(name,configs):

    # New teacher version => old problematic caches cannot be loaded.
    path=out/f"{name}_{cfg.teacher_version}_{tag}.pkl"

    if path.exists():
        print("Loading",path)
        with open(path,"rb") as f:
            return pickle.load(f)

    x=make_dataset(configs,name)

    with open(path,"wb") as f:
        pickle.dump(x,f)

    return x


train=load_or_make(
    "train",
    design(cfg.n_train,cfg.train_N,cfg.seed+1)
)

valid=load_or_make(
    "validation",
    design(cfg.n_val,cfg.train_N,cfg.seed+2)
)

test_seen=load_or_make(
    "test_seen",
    design(cfg.n_test_seen,cfg.train_N,cfg.seed+3)
)

test_interp=load_or_make(
    "test_interp",
    design(cfg.n_test_interp,cfg.interp_N,cfg.seed+4)
)


# =====================================================================================
# 7. TARGET AUDIT
# =====================================================================================

def audit(records,name):

    bad_p=[
        r for r in records
        if (
            not np.all(np.isfinite(r.p))
            or
            abs(r.p.sum()-1.)>1e-6
        )
    ]

    if bad_p:
        raise RuntimeError(f"Invalid probability targets in {name}.")

    print(f"\n{name}")

    for N in sorted({r.N for r in records}):
        x=[r for r in records if r.N==N]

        print(
            f"N={N:3d} | n={len(x):3d} | "
            f"i0=1={sum(r.i0==1 for r in x):3d} | "
            f"valid tau={sum(r.tau_valid for r in x):3d}/{len(x):3d}"
        )


print("\n"+"="*108)
print("EXACT-TARGET AUDIT")
print("="*108)

audit(train,"TRAIN")
audit(valid,"VALIDATION")
audit(test_seen,"TEST-SEEN")
audit(test_interp,"TEST-INTERPOLATION")

if not any(r.tau_valid for r in train):
    raise RuntimeError("No valid extinction-time training targets.")


# =====================================================================================
# 8. NETWORKS
# =====================================================================================

def mlp(din,dout):
    L=[]; d=din

    for _ in range(cfg.depth):
        L += [nn.Linear(d,cfg.width),nn.SiLU()]
        d=cfg.width

    L += [nn.Linear(d,dout)]
    return nn.Sequential(*L)


class HazardNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=mlp(6,1)

    def forward(self,x):
        return torch.sigmoid(
            self.net(x).squeeze(-1)
        )


class TauNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=mlp(5,2)

    def forward(self,x):
        return torch.nn.functional.softplus(
            self.net(x)
        )


def xtau(r):
    return torch.tensor(
        [
            r.beta,r.gamma,r.omega,
            r.N/N_scale,r.i0/r.N
        ],
        dtype=torch.float32,
        device=device
    )


def xhaz(r):
    c=torch.arange(
        r.N+1,
        dtype=torch.float32,
        device=device
    )

    n=r.N+1

    return torch.column_stack([
        torch.full((n,),r.beta,device=device),
        torch.full((n,),r.gamma,device=device),
        torch.full((n,),r.omega,device=device),
        torch.full((n,),r.N/N_scale,device=device),
        torch.full((n,),r.i0/r.N,device=device),
        c/r.N
    ])


def reconstruct(h):
    before=torch.cat([
        torch.ones(1,device=h.device),
        torch.cumprod(1-h[:-1],0)
    ])

    return torch.cat([
        before*h,
        torch.prod(1-h).reshape(1)
    ])


def phat_tensor(model,r):
    return reconstruct(
        model(xhaz(r))
    )


def tau_target(r):

    if not r.tau_valid:
        raise RuntimeError(
            "tau_target called for unresolved exact tau target."
        )

    raw=np.asarray(
        [r.mean_tau,r.var_tau],
        dtype=np.float64
    )

    if (
        not np.all(np.isfinite(raw))
        or raw[0]<=0
        or raw[1]<0
    ):
        raise RuntimeError(
            f"Invalid tau target: N={r.N}, i0={r.i0}, {raw}"
        )

    with np.errstate(invalid="raise",over="raise"):
        z=np.log1p(raw)

    return torch.tensor(
        z,
        dtype=torch.float32,
        device=device
    )


# =====================================================================================
# 9. LOSS
# =====================================================================================

def batch_loss(records,ix,hnet,tnet):

    LP=[]
    LT=[]

    for j in ix:
        r=records[int(j)]

        # Distributional target is always available.
        p=torch.tensor(
            r.p,
            dtype=torch.float32,
            device=device
        )

        LP.append(
            torch.sum(
                (
                    phat_tensor(hnet,r)
                    -
                    p
                )**2
            )
        )

        # Tau target contributes only when the exact sparse-LU result is admissible.
        if r.tau_valid:

            z=tau_target(r)

            zh=tnet(
                xtau(r).unsqueeze(0)
            ).squeeze(0)

            LT.append(
                torch.sum(
                    (zh-z)**2
                    /
                    (1+z*z)
                )
            )

    lp=torch.stack(LP).mean()

    if LT:
        lt=torch.stack(LT).mean()
    else:
        lt=torch.zeros(
            (),
            dtype=torch.float32,
            device=device
        )

    return (
        lp+cfg.lambda_tau*lt,
        lp,
        lt
    )


# =====================================================================================
# 10. TRAINING
# =====================================================================================

@torch.no_grad()
def validation_loss(hnet,tnet):
    hnet.eval(); tnet.eval()

    L,_,_=batch_loss(
        valid,
        np.arange(len(valid)),
        hnet,
        tnet
    )

    return float(L.item())


def train_emulator(records,seed,verbose=False):

    seed_all(seed)

    n_tau=sum(r.tau_valid for r in records)

    if n_tau==0:
        raise RuntimeError(
            f"R={len(records)} contains no numerically resolved tau targets."
        )

    hnet=HazardNet().to(device)
    tnet=TauNet().to(device)

    pars=list(hnet.parameters())+list(tnet.parameters())

    opt=torch.optim.AdamW(
        pars,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=.5,
        patience=15
    )

    rng=np.random.default_rng(seed+7011)

    best=np.inf
    bh=bt=None
    wait=0

    hist={
        "joint":[],
        "dist":[],
        "tau":[],
        "val":[]
    }

    t0=time.perf_counter()

    for epoch in range(1,cfg.epochs+1):

        hnet.train(); tnet.train()
        perm=rng.permutation(len(records))

        J=[];P=[];T=[]

        for s in range(0,len(perm),cfg.batch_size):

            ix=perm[s:s+cfg.batch_size]
            opt.zero_grad()

            L,Lp,Lt=batch_loss(
                records,ix,hnet,tnet
            )

            if not torch.isfinite(L):
                raise RuntimeError(
                    f"Non-finite loss: R={len(records)}, epoch={epoch}."
                )

            L.backward()

            torch.nn.utils.clip_grad_norm_(
                pars,
                cfg.grad_clip
            )

            opt.step()

            J.append(L.item())
            P.append(Lp.item())
            T.append(Lt.item())

        j,p,t=np.mean(J),np.mean(P),np.mean(T)
        v=validation_loss(hnet,tnet)

        if not np.isfinite(v):
            raise RuntimeError(
                f"Non-finite validation loss: R={len(records)}, epoch={epoch}."
            )

        sch.step(v)

        hist["joint"].append(j)
        hist["dist"].append(p)
        hist["tau"].append(t)
        hist["val"].append(v)

        if bh is None or v<best-cfg.min_delta:
            best=v
            bh=copy.deepcopy(hnet.state_dict())
            bt=copy.deepcopy(tnet.state_dict())
            wait=0
        else:
            wait+=1

        if verbose and (epoch==1 or epoch%20==0):
            print(
                f"R={len(records):4d} | epoch={epoch:4d} | "
                f"joint={j:.4e} | dist={p:.4e} | "
                f"tau={t:.4e} | val={v:.4e}"
            )

        if wait>=cfg.patience:
            break

    hnet.load_state_dict(bh)
    tnet.load_state_dict(bt)

    return {
        "hazard":hnet,
        "tau":tnet,
        "history":hist,
        "training_time":time.perf_counter()-t0,
        "best_validation_loss":best,
        "n_tau_train":n_tau
    }


# =====================================================================================
# 11. EVALUATION
# =====================================================================================

def tail(p):
    return np.flip(
        np.cumsum(
            np.flip(p[1:])
        )
    )


@torch.no_grad()
def predict(r,hnet,tnet):

    hnet.eval(); tnet.eval()

    p=phat_tensor(
        hnet,r
    ).cpu().numpy()

    z=(
        tnet(
            xtau(r).unsqueeze(0)
        )
        .squeeze(0)
        .cpu()
        .numpy()
        .astype(np.float64)
    )

    moments=np.expm1(
        np.clip(z,0.,700.)
    )

    return p,float(moments[0]),float(moments[1])


def evaluate(records,hnet,tnet,split):

    ans=[]

    for j,r in enumerate(records):

        p,m,v=predict(r,hnet,tnet)

        rho=tail(r.p)
        rhoh=tail(p)

        pos=r.p>0
        ps=np.clip(p,cfg.kl_eps,1.)

        row={
            "index":j,
            "split":split,
            "N":r.N,
            "i0":r.i0,
            "beta":r.beta,
            "gamma":r.gamma,
            "omega":r.omega,
            "tau_valid":r.tau_valid,

            "E2":
                float(np.linalg.norm(p-r.p)),

            "E_rho":
                float(np.max(np.abs(rhoh-rho))),

            "E_overflow":
                float(abs(p[-1]-r.p[-1])),

            "KL":
                float(
                    np.sum(
                        r.p[pos]
                        *
                        np.log(r.p[pos]/ps[pos])
                    )
                ),

            "exact_p":r.p,
            "pred_p":p,
            "exact_tail":rho,
            "pred_tail":rhoh
        }

        if r.tau_valid:
            row["mean_tau_relative_error"]=float(
                abs(m-r.mean_tau)/r.mean_tau
            )

            row["var_tau_relative_error"]=float(
                abs(v-r.var_tau)/max(r.var_tau,1e-300)
            )

        else:
            row["mean_tau_relative_error"]=np.nan
            row["var_tau_relative_error"]=np.nan

        ans.append(row)

    return ans


METRICS=[
    "E2","E_rho","E_overflow","KL",
    "mean_tau_relative_error","var_tau_relative_error"
]


def finite_values(results,key):
    x=np.asarray(
        [r[key] for r in results],
        dtype=float
    )
    return x[np.isfinite(x)]


def med(results,key):
    x=finite_values(results,key)
    return float(np.median(x)) if len(x) else np.nan


# =====================================================================================
# 12. BALANCED NESTED TRAINING ORDER
# =====================================================================================

def balanced_order(records,seed):

    rng=np.random.default_rng(seed)
    groups={}

    for N in cfg.train_N:
        for flag in (0,1):

            x=np.asarray([
                j for j,r in enumerate(records)
                if r.N==N and int(r.i0==1)==flag
            ],dtype=int)

            rng.shuffle(x)
            groups[(N,flag)]=list(x)

    ptr={k:0 for k in groups}
    usedN={N:0 for N in cfg.train_N}
    used1=0
    order=[]

    for position in range(len(records)):

        target1=cfg.i0_one_fraction*(position+1)
        pref=1 if used1<target1 else 0
        selected=None

        for flag in (pref,1-pref):

            candidates=[
                N for N in cfg.train_N
                if ptr[(N,flag)]<len(groups[(N,flag)])
            ]

            if candidates:

                mn=min(usedN[N] for N in candidates)
                candidates=[
                    N for N in candidates
                    if usedN[N]==mn
                ]

                N=int(rng.choice(candidates))
                selected=(N,flag)
                break

        if selected is None:
            raise RuntimeError(
                "Could not construct balanced nested ordering."
            )

        N,flag=selected

        j=groups[(N,flag)][ptr[(N,flag)]]

        ptr[(N,flag)]+=1
        usedN[N]+=1
        used1+=flag

        order.append(j)

    return np.asarray(order,dtype=int)


# =====================================================================================
# 13. FIXED DISTRIBUTIONAL SHOWCASES
# =====================================================================================

def entropy(p):
    z=p[p>0]
    return float(-np.sum(z*np.log(z)))


def bimodality(p):

    z=p[:-1]

    if len(z)<3 or z.max()<=0:
        return 0.

    peaks=[
        j for j in range(len(z))
        if z[j]>=(z[j-1] if j else -np.inf)
        and z[j]>=(z[j+1] if j<len(z)-1 else -np.inf)
        and z[j]>=.03*z.max()
    ]

    if len(peaks)<2:
        return 0.

    a,b=sorted(
        peaks,
        key=lambda j:z[j],
        reverse=True
    )[:2]

    return float(
        min(z[a],z[b])/max(z[a],z[b])
        *
        abs(a-b)/max(len(z)-1,1)
    )


large500=[
    r for r in test_seen
    if r.N==500 and r.i0==1
]

if not large500:
    large500=[
        r for r in test_seen
        if r.N==500
    ]

case1=max(large500,key=lambda r:entropy(r.p))
case2=max(test_interp,key=lambda r:bimodality(r.p))

SHOWCASES=[
    (r"$N=500,\ i_0=1$",case1),
    ("Held-out interpolation",case2)
]


# =====================================================================================
# 14. MAIN LEARNING-CURVE EXPERIMENT
# =====================================================================================

rows=[]
gallery={lab:{} for lab,_ in SHOWCASES}


for rep in range(REPEATS):

    print("\n"+"="*108)
    print(f"LEARNING-CURVE REPETITION {rep+1}/{REPEATS}")
    print("="*108)

    order=balanced_order(
        train,
        cfg.seed+20000+rep
    )

    for R in R_VALUES:

        subset=[
            train[int(j)]
            for j in order[:R]
        ]

        counts={
            N:sum(r.N==N for r in subset)
            for N in cfg.train_N
        }

        n1=sum(r.i0==1 for r in subset)
        ntau=sum(r.tau_valid for r in subset)

        print("\n"+"-"*108)
        print(f"Repetition {rep+1} | R={R}")
        print("N counts:",counts)
        print(
            f"i0=1: {n1}/{R} ({n1/R:.1%}) | "
            f"valid tau: {ntau}/{R}"
        )

        fit=train_emulator(
            subset,
            seed=cfg.seed+30000+1000*rep+R
        )

        h,t=fit["hazard"],fit["tau"]

        seen=evaluate(
            test_seen,h,t,"seen"
        )

        interp=evaluate(
            test_interp,h,t,"interpolation"
        )

        interp1=[
            x for x in interp
            if x["i0"]==1
        ]

        interp2=[
            x for x in interp
            if x["i0"]>1
        ]

        for split,res in [
            ("seen",seen),
            ("interpolation",interp),
            ("interpolation_i0_1",interp1),
            ("interpolation_i0_gt1",interp2)
        ]:

            row={
                "rep":rep,
                "R":R,
                "split":split,
                "training_time":fit["training_time"],
                "best_validation_loss":fit["best_validation_loss"],
                "n_tau_train":fit["n_tau_train"],
                "n_tau_eval":sum(x["tau_valid"] for x in res)
            }

            for k in METRICS:
                row[k]=med(res,k)

            rows.append(row)

        if rep==0 and R in GALLERY_R:

            for lab,r in SHOWCASES:
                gallery[lab][R]=evaluate(
                    [r],h,t,"gallery"
                )[0]

        print(
            f"interp E2={med(interp,'E2'):.4e} | "
            f"E_rho={med(interp,'E_rho'):.4e} | "
            f"rel E(tau)={med(interp,'mean_tau_relative_error'):.4e} | "
            f"rel Var(tau)={med(interp,'var_tau_relative_error'):.4e}"
        )

        del fit,h,t,seen,interp,interp1,interp2

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# =====================================================================================
# 15. CURVES ACROSS REPETITIONS
# =====================================================================================

def curve(split,key):

    M=[];Q1=[];Q3=[]

    for R in R_VALUES:

        x=np.asarray([
            z[key]
            for z in rows
            if (
                z["R"]==R
                and
                z["split"]==split
                and
                np.isfinite(z[key])
            )
        ])

        if len(x):
            M.append(np.median(x))
            Q1.append(np.quantile(x,.25))
            Q3.append(np.quantile(x,.75))
        else:
            M.append(np.nan)
            Q1.append(np.nan)
            Q3.append(np.nan)

    return (
        np.asarray(R_VALUES),
        np.asarray(M),
        np.asarray(Q1),
        np.asarray(Q3)
    )


print("\n"+"="*115)
print("HELD-OUT POPULATION-SIZE INTERPOLATION SUMMARY")
print("="*115)

curves={
    k:curve("interpolation",k)[1]
    for k in METRICS
}

print(
    f"{'R':>6s} | {'E2':>11s} | {'E_rho':>11s} | "
    f"{'E_over':>11s} | {'KL':>11s} | "
    f"{'Rel E(tau)':>12s} | {'Rel Var(tau)':>14s}"
)

print("-"*115)

for j,R in enumerate(R_VALUES):

    print(
        f"{R:6d} | "
        f"{curves['E2'][j]:11.4e} | "
        f"{curves['E_rho'][j]:11.4e} | "
        f"{curves['E_overflow'][j]:11.4e} | "
        f"{curves['KL'][j]:11.4e} | "
        f"{curves['mean_tau_relative_error'][j]:12.4e} | "
        f"{curves['var_tau_relative_error'][j]:14.4e}"
    )


# =====================================================================================
# 16. PLOTS
# =====================================================================================

plt.rcParams.update({
    "font.size":10.5,
    "axes.spines.top":False,
    "axes.spines.right":False
})


# ----- Figure 1: seen vs interpolation -----

fig,axs=plt.subplots(2,2,figsize=(13,9))

spec=[
    ("E2",r"Median $E_2$","(A) Distributional error"),
    ("E_rho",r"Median $E_\rho$","(B) Tail-risk error"),
    ("mean_tau_relative_error","Median relative error",r"(C) $E(\tau)$"),
    ("var_tau_relative_error","Median relative error",r"(D) $\mathrm{Var}(\tau)$")
]

for ax,(key,ylab,title) in zip(axs.flat,spec):

    for split,label,col,mk in [
        ("seen","Training-grid $N$","#555555","o"),
        ("interpolation","Held-out interpolation $N$","#0072B2","D")
    ]:

        R,m,l,u=curve(split,key)
        ok=np.isfinite(m)

        ax.plot(
            R[ok],
            np.maximum(m[ok],1e-12),
            color=col,marker=mk,lw=2,label=label
        )

        ax.fill_between(
            R[ok],
            np.maximum(l[ok],1e-12),
            np.maximum(u[ok],1e-12),
            color=col,alpha=.12
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xticks(R_VALUES)
    ax.set_xticklabels([str(x) for x in R_VALUES])
    ax.set_xlabel(r"Exact training configurations $R$")
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.grid(alpha=.15)
    ax.legend(frameon=False)

fig.suptitle(
    "Accuracy Versus Exact-Teacher Training-Set Size",
    fontsize=15
)

plt.tight_layout()
plt.savefig(
    out/"figure_5_1B_learning_curves.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)
plt.show()


# ----- Figure 2: i0=1 vs i0>1 -----

fig,axs=plt.subplots(2,3,figsize=(17,9))

spec2=[
    ("E2",r"$E_2$","(A) Distributional error"),
    ("E_rho",r"$E_\rho$","(B) Tail-risk error"),
    ("E_overflow","Overflow error","(C) Overflow risk"),
    ("KL","KL divergence","(D) KL divergence"),
    ("mean_tau_relative_error","Relative error",r"(E) $E(\tau)$"),
    ("var_tau_relative_error","Relative error",r"(F) $\mathrm{Var}(\tau)$")
]

for ax,(key,ylab,title) in zip(axs.flat,spec2):

    for split,label,col,mk in [
        ("interpolation_i0_1",r"$i_0=1$","#0072B2","o"),
        ("interpolation_i0_gt1",r"$i_0>1$","#E69F00","s")
    ]:

        R,m,l,u=curve(split,key)
        ok=np.isfinite(m)

        ax.plot(
            R[ok],
            np.maximum(m[ok],1e-12),
            color=col,marker=mk,lw=2,label=label
        )

        ax.fill_between(
            R[ok],
            np.maximum(l[ok],1e-12),
            np.maximum(u[ok],1e-12),
            color=col,alpha=.12
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xticks(R_VALUES)
    ax.set_xticklabels([str(x) for x in R_VALUES])
    ax.set_xlabel(r"Exact training configurations $R$")
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.grid(alpha=.15)
    ax.legend(frameon=False)

fig.suptitle(
    r"Interpolation Accuracy for $i_0=1$ and $i_0>1$",
    fontsize=15
)

plt.tight_layout()
plt.savefig(
    out/"figure_5_1B_i0_learning_curves.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)
plt.show()


# ----- Figure 3: PMFs as R grows -----

fig,axs=plt.subplots(
    2,len(GALLERY_R),
    figsize=(19,8),
    squeeze=False
)

for row,(lab,_) in enumerate(SHOWCASES):

    for col,R in enumerate(GALLERY_R):

        ax=axs[row,col]
        z=gallery[lab][R]
        c=np.arange(z["N"]+1)

        ax.bar(
            c,z["exact_p"][:-1],
            width=.85,color=".82",
            label="Exact Markovian"
        )

        ax.plot(
            c,z["pred_p"][:-1],
            color="#D55E00",
            lw=1.8,
            label="Neural emulator"
        )

        ax.set_title(
            f"{lab}, $R={R}$\n"
            f"$N={z['N']}$, $i_0={z['i0']}$, "
            rf"$E_2={z['E2']:.3f}$, "
            rf"$E_\rho={z['E_rho']:.3f}$"
        )

        ax.set_xlabel("Infection count $c$")
        ax.set_ylabel("Probability mass")

        if row==0 and col==0:
            ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(
    out/"figure_5_1B_pmf_vs_R.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)
plt.show()


# ----- Figure 4: tails as R grows -----

fig,axs=plt.subplots(
    2,len(GALLERY_R),
    figsize=(19,8),
    squeeze=False
)

for row,(lab,_) in enumerate(SHOWCASES):

    for col,R in enumerate(GALLERY_R):

        ax=axs[row,col]
        z=gallery[lab][R]
        c=np.arange(z["N"]+1)

        ax.plot(
            c,z["exact_tail"],
            color="black",
            lw=2,
            label="Exact Markovian"
        )

        ax.plot(
            c,z["pred_tail"],
            "--",
            color="#0072B2",
            lw=1.8,
            label="Neural emulator"
        )

        ax.set_ylim(-.01,1.01)

        ax.set_title(
            f"{lab}, $R={R}$\n"
            f"$N={z['N']}$, $i_0={z['i0']}$, "
            rf"$E_\rho={z['E_rho']:.3f}$"
        )

        ax.set_xlabel("Threshold $c$")
        ax.set_ylabel(r"$P(C>c)$")

        if row==0 and col==0:
            ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(
    out/"figure_5_1B_tail_vs_R.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)
plt.show()


# =====================================================================================
# 17. SAVE
# =====================================================================================

result_path=out/"section_5_1B_results.pkl"

with open(result_path,"wb") as f:
    pickle.dump({
        "config":asdict(cfg),
        "R_VALUES":R_VALUES,
        "REPEATS":REPEATS,
        "rows":rows,
        "gallery":gallery,

        "tau_resolution":{
            "train":(sum(r.tau_valid for r in train),len(train)),
            "validation":(sum(r.tau_valid for r in valid),len(valid)),
            "test_seen":(sum(r.tau_valid for r in test_seen),len(test_seen)),
            "test_interp":(sum(r.tau_valid for r in test_interp),len(test_interp))
        }
    },f)


print("\n"+"="*108)
print("EXPERIMENT 5.1-B COMPLETE")
print("="*108)

print("Training N:",cfg.train_N)
print("Held-out interpolation N:",cfg.interp_N)
print("R:",R_VALUES)
print("No extrapolation is used.")

print("\nNO inverse of T or D0 was computed.")
print("All Markovian calculations used sparse LU linear solves.")

print(
    f"\nResolved tau targets — training: "
    f"{sum(r.tau_valid for r in train)}/{len(train)}"
)

print(
    f"Resolved tau targets — validation: "
    f"{sum(r.tau_valid for r in valid)}/{len(valid)}"
)

print(
    f"Resolved tau targets — seen test: "
    f"{sum(r.tau_valid for r in test_seen)}/{len(test_seen)}"
)

print(
    f"Resolved tau targets — interpolation test: "
    f"{sum(r.tau_valid for r in test_interp)}/{len(test_interp)}"
)

print("\nResults:",result_path.resolve())
print("="*108)

EXPERIMENT 5.1-B — SPARSE-LU INTERPOLATION STUDY
Device: cpu
Training N: (20, 50, 100, 200, 500)
Held-out interpolation N: (35, 75, 150, 350)
R: (10, 20, 50, 100, 200, 400, 800)  | repetitions: 3
N=500 transient states: 125,250
NO MATRIX INVERSE IS COMPUTED.
[train       ]    5/ 800 | N= 50, i0=  8 | tau unresolved=  0 | 17.1s
[train       ]   10/ 800 | N= 20, i0=  1 | tau unresolved=  0 | 33.6s
[train       ]   15/ 800 | N= 20, i0=  1 | tau unresolved=  0 | 49.3s
[train       ]   20/ 800 | N=200, i0= 40 | tau unresolved=  0 | 50.0s
[train       ]   25/ 800 | N=100, i0=  4 | tau unresolved=  0 | 50.6s
[train       ]   30/ 800 | N= 50, i0=  7 | tau unresolved=  0 | 59.3s
[train       ]   35/ 800 | N= 20, i0=  2 | tau unresolved=  0 | 59.3s
[train       ]   40/ 800 | N=200, i0= 20 | tau unresolved=  0 | 67.3s
[train       ]   45/ 800 | N=100, i0= 10 | tau unresolved=  0 | 75.7s
[train       ]   50/ 800 | N=200, i0= 39 | tau unresolved=  0 | 77.3s
[train       ]   55/ 800 | N= 20, i0=  2 